In [1]:
%pip install lightsim2grid accelerate grid2op scikit-learn transformers xgboost -q


# %%== CELL 2: MAMBA INSTALL ==================================
%pip install torch==2.10.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
%pip install https://github.com/state-spaces/mamba/releases/download/v2.3.2.post1/mamba_ssm-2.3.2.post1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl --force-reinstall --no-deps

Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://download.pytorch.org/whl/cu128
  Using cached https://download-r2.pytorch.org/whl/cu128/torch-2.10.0%2Bcu128-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (30 kB)
  Using cached https://download.pytorch.org/whl/cu128/cuda_bindings-12.9.4-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (2.6 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 48.5 MB/s  0:00:07:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.1/139.1 MB 55.2 MB/s  0:00:02:00:0100:01
  Using cached https://download-r2.pytorch.org/whl/triton-3.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (1.7 kB)
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
  Using cached https://download-r2.pytorch.org/whl/cu128/torchvision-0.26.0%2Bcu128-cp312-cp312-manylinux_2_28_x86_64

In [2]:
%pip install https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.1.post4/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.6/287.6 MB 71.8 MB/s  0:00:04:00:0100:01
  Using cached ninja-1.13.0-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.1 kB)
Using cached ninja-1.13.0-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (180 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [causal-conv1d]0m [causal-conv1d]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mamba-ssm 2.3.2.post1 requires apache-tvm-ffi<=0.1.9, which is not installed.
mamba-ssm 2.3.2.post1 requires einops, which is not installed.
mamba-ssm 2.3.2.post1 requires quack-kernels>=0.3.4, which is not installed.
mamba-ssm 2.3.2.post1 requires tilelang==0.1.8, which is not installed.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from mamba_ssm import Mamba3
import torch
model = Mamba3(d_model=128, d_state=32, headdim=64, is_mimo=False).cuda()
x = torch.randn(2, 128, 128).cuda()
y = model(x)
print(f"✅ Mamba working: {y.shape}")
del model, x, y

✅ Mamba working: torch.Size([2, 128, 128])


Chronic Extraction Method (already done previously and now loaded as a zip file)

In [ ]:
def extract_sequence_features(obs):
    """Tier 1: Dense array for the sequence model."""
    return np.concatenate([
        obs.rho,                                    # (N_LINE,)
        obs.v_or,                                   # (N_LINE,)
        obs.v_ex,                                   # (N_LINE,)
        obs.p_or,                                   # (N_LINE,)
        obs.q_or,                                   # (N_LINE,)
        obs.line_status.astype(np.float32),         # (N_LINE,)
        obs.timestep_overflow.astype(np.float32),   # (N_LINE,)
        obs.load_p,                                 # (N_LOAD,)
        obs.gen_p,                                  # (N_GEN,)
        obs.topo_vect.astype(np.float32),           # (DIM_TOPO,)
    ]).astype(np.float32)


def extract_tabular_features(obs):
    """Tier 2: Scalar summaries for XGBoost."""
    # Weather proxies
    solar_total_cap = gen_pmax[solar_mask].sum()
    wind_total_cap  = gen_pmax[wind_mask].sum()
    solar_cf = obs.gen_p[solar_mask].sum() / (solar_total_cap + 1e-8) if solar_mask.any() else 0.0
    wind_cf  = obs.gen_p[wind_mask].sum()  / (wind_total_cap  + 1e-8) if wind_mask.any() else 0.0

    # Voltages (combine both ends)
    all_v = np.concatenate([obs.v_or, obs.v_ex])

    # Handle potentially empty arrays from specific grid configurations
    def safe_min(arr, default=-1.0):
        return arr.min() if arr.size > 0 else default

    return np.array([
        # Weather proxies
        solar_cf,
        wind_cf,
        # Grid stress
        obs.rho.max(),
        obs.rho.mean(),
        obs.rho.std(),
        (obs.rho >= 0.85).sum(),                          # n_overloaded
        (obs.rho >= 0.95).sum(),                          # n_critical
        all_v.min(),                                      # min_voltage
        all_v.mean(),                                     # mean_voltage
        (~obs.line_status).sum(),                         # n_disconnected
        # Operational state
        (obs.time_before_cooldown_line > 0).sum(),        # n_lines_cooldown
        obs.time_before_cooldown_line.max() if obs.time_before_cooldown_line.size > 0 else 0.0,
        (obs.time_before_cooldown_sub > 0).sum(),         # n_subs_cooldown
        (obs.time_next_maintenance == 0).sum(),           # n_lines_in_maintenance
        safe_min(obs.time_since_last_attack) if hasattr(obs, 'time_since_last_attack') else -1,
        # Storage
        obs.storage_charge.mean() if obs.storage_charge is not None and obs.storage_charge.size > 0 else 0.0,
        obs.storage_power.sum()   if obs.storage_power is not None and obs.storage_power.size > 0 else 0.0,
        # Load/gen balance
        obs.load_p.sum(),
        obs.load_p.std(),
        obs.load_p.max() / (obs.load_p.mean() + 1e-8),
        obs.gen_p.sum(),
        obs.load_p.sum() / (obs.gen_p.sum() + 1e-8),     # load_gen_ratio
        # Temporal
        np.sin(2 * np.pi * obs.hour_of_day / 24),
        np.cos(2 * np.pi * obs.hour_of_day / 24),
        obs.day_of_week,
        obs.month,
    ], dtype=np.float32)


TABULAR_NAMES = [
    'solar_cf', 'wind_cf',
    'max_rho', 'mean_rho', 'std_rho', 'n_overloaded', 'n_critical',
    'min_voltage', 'mean_voltage', 'n_disconnected',
    'n_lines_cooldown', 'max_cooldown', 'n_subs_cooldown',
    'n_lines_maintenance', 'time_since_attack',
    'storage_soc_mean', 'storage_power_total',
    'total_load', 'load_std', 'peak_load_ratio',
    'total_gen', 'load_gen_ratio',
    'hour_sin', 'hour_cos', 'day_of_week', 'month',
]


def assign_labels(obs, done):
    """Multi-label: [thermal, voltage, blackout]."""
    return np.array([
        float((obs.rho >= 0.85).any()),                                   # Thermal
        float((obs.v_or < 0.90).any() or (obs.v_ex < 0.90).any()),       # Voltage
        float(done),                                                       # Blackout
    ], dtype=np.float32)

In [ ]:
import gc

n_chronics = len(env.chronics_handler.real_data.subpaths)
print(f"Total chronics available: {n_chronics}")

TARGET_STEPS = 1_200_000
SAVE_DIR = 'root/chronics'
os.makedirs(SAVE_DIR, exist_ok=True)

total_steps = 0
chronic_meta = []  # Track what we saved

for c_id in tqdm(range(n_chronics), desc="Chronics"):
    env.set_id(c_id)
    obs = env.reset()
    done = False

    chronic_seq = []
    chronic_tab = []
    chronic_lbl = []

    while not done:
        chronic_seq.append(extract_sequence_features(obs))
        chronic_tab.append(extract_tabular_features(obs))

        action = env.action_space({})
        next_obs, reward, done, info = env.step(action)
        chronic_lbl.append(assign_labels(next_obs, done))
        obs = next_obs

    
    seq_arr = np.array(chronic_seq, dtype=np.float32)
    tab_arr = np.array(chronic_tab, dtype=np.float32)
    lbl_arr = np.array(chronic_lbl, dtype=np.float32)

    # Replace NaN/Inf
    seq_arr = np.nan_to_num(seq_arr, nan=0.0, posinf=0.0, neginf=0.0)
    tab_arr = np.nan_to_num(tab_arr, nan=0.0, posinf=0.0, neginf=0.0)

    np.save(f'{SAVE_DIR}/seq_{c_id:04d}.npy', seq_arr)
    np.save(f'{SAVE_DIR}/tab_{c_id:04d}.npy', tab_arr)
    np.save(f'{SAVE_DIR}/lbl_{c_id:04d}.npy', lbl_arr)

    n_steps = len(chronic_seq)
    total_steps += n_steps
    chronic_meta.append({'id': c_id, 'n_steps': n_steps})

    
    del chronic_seq, chronic_tab, chronic_lbl, seq_arr, tab_arr, lbl_arr
    gc.collect()

    if (c_id + 1) % 20 == 0:
        print(f"   {total_steps:,} steps collected so far")

    if total_steps >= TARGET_STEPS:
        print(f"\nReached {total_steps:,} steps after {c_id+1} chronics. Stopping.")
        break

    

In [ ]:
import grid2op
import numpy as np
import pandas as pd
from tqdm import tqdm
import os, json

env = grid2op.make("l2rpn_wcci_2020")
obs = env.reset()

# ── Verify grid dimensions ──
N_LINE = env.n_line
N_LOAD = env.n_load
N_GEN  = env.n_gen
DIM_TOPO = obs.topo_vect.shape[0]

print(f"Lines: {N_LINE}, Loads: {N_LOAD}, Gens: {N_GEN}, Topo dim: {DIM_TOPO}")

# ── Identify renewable generators ──
gen_type = list(env.gen_type)
gen_pmax = env.gen_pmax

solar_mask = np.array([t == 'solar' for t in gen_type])
wind_mask  = np.array([t == 'wind'  for t in gen_type])

print(f"Solar gens: {solar_mask.sum()}, Wind gens: {wind_mask.sum()}")
print(f"Gen types: {pd.Series(gen_type).value_counts().to_dict()}")

# ── Feature dimensions ──
SEQ_DIM = N_LINE * 7 + N_LOAD + N_GEN + DIM_TOPO
# 7 per line: rho, v_or, v_ex, p_or, q_or, line_status, timestep_overflow
print(f"\nSequence feature dim per timestep: {SEQ_DIM}")

# Also used in previous runs, giving the following output:

Lines: 59, Loads: 37, Gens: 22, Topo dim: 177
Solar gens: 8, Wind gens: 4
Gen types: {np.str_('thermal'): 8, np.str_('solar'): 8, np.str_('wind'): 4, np.str_('hydro'): 1, np.str_('nuclear'): 1}

Sequence feature dim per timestep: 649

In [ ]:
# %%== CELL 1: DOWNLOAD & UNZIP DATASET FROM DRIVE ======================
import os
import zipfile

try:
    import gdown
except ImportError:
    os.system('pip install gdown')
    import gdown

URL_GRID2OP = os.environ.get["URL_GRID2OP"]
URL_CHRONICS = os.environ.get["URL_CHRONICS"]

print("Downloading grid2op.zip from Drive...")
gdown.download(URL_GRID2OP, 'grid2op.zip', quiet=False)

print("Downloading gridchronics.zip from Drive...")
gdown.download(URL_CHRONICS, 'gridchronics.zip', quiet=False)

os.makedirs("./dataset", exist_ok=True)

print("Extracting grid2op.zip...")
with zipfile.ZipFile('grid2op.zip', 'r') as zip_ref:
    zip_ref.extractall("/workspace/dataset")

print("Extracting gridchronics.zip...")
with zipfile.ZipFile('gridchronics.zip', 'r') as zip_ref:
    zip_ref.extractall("/workspace/dataset")

print("✅ Download and Extraction Complete!")

Downloading...
From (original): https://drive.google.com/uc?id=1PlI9U7jHLX5AoHeJcgzOFB6vOi1zUxcy
From (redirected): https://drive.google.com/uc?id=1PlI9U7jHLX5AoHeJcgzOFB6vOi1zUxcy&confirm=t&uuid=4002a05c-da57-4480-b678-cb4992bb42e8
To: /workspace/grid2op.zip
100%|██████████| 4.53G/4.53G [01:58<00:00, 38.1MB/s]


Downloading...
From (original): https://drive.google.com/uc?id=1IhRk1ZVeham1Gx-nUcH68VDvOrJvVWdD
From (redirected): https://drive.google.com/uc?id=1IhRk1ZVeham1Gx-nUcH68VDvOrJvVWdD&confirm=t&uuid=30dc5a1c-f3b4-4179-873e-394b80b6330d
To: /workspace/gridchronics.zip
100%|██████████| 1.17G/1.17G [02:14<00:00, 8.68MB/s]


Extracting grid2op.zip...
Extracting gridchronics.zip...
✅ Download and Extraction Complete!


In [2]:
%pip install einops

  Using cached einops-0.8.2-py3-none-any.whl.metadata (13 kB)
Using cached einops-0.8.2-py3-none-any.whl (65 kB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mamba-ssm 2.3.2.post1 requires apache-tvm-ffi<=0.1.9, which is not installed.
mamba-ssm 2.3.2.post1 requires quack-kernels>=0.3.4, which is not installed.
mamba-ssm 2.3.2.post1 requires tilelang==0.1.8, which is not installed.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import json
import time
import math
import random
from mamba_ssm import Mamba2
import shutil
from pathlib import Path
from typing import Tuple, Dict, Any, List

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from accelerate import Accelerator
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, accuracy_score

# ---------- CONFIG ----------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


CHRONICS_DIR = "/workspace/dataset/content/root/chronics" 
SAVE_DIR = "/workspace/grid_models"

os.makedirs(SAVE_DIR, exist_ok=True)

CONFIG = {
    'data_dir': CHRONICS_DIR,
    'save_dir': SAVE_DIR,
    'window_size': 128,
    'stride': 16,
    'pred_horizon': 4,        # forecast horizon (timesteps ahead)
    'd_model': 128,
    'n_layers': 4,
    'd_state': 32,
    'batch_size': 256,
    'grad_accum': 1,
    'lr': 3e-4,
    'weight_decay': 1e-2,
    'epochs': 20,
    'patience': 5,
    'lam_kirchhoff': 0.01,
    'lam_thermal': 0.01,
    'random_split_seed': SEED,
    'train_frac': 0.70,
    'val_frac': 0.15,
    'test_frac': 0.15,
}
# ---------- END CONFIG ----------


def save_meta(meta: Dict[str, Any], path: str) -> None:
    with open(path, 'w') as fh:
        json.dump(meta, fh, indent=2)


def load_chronics_meta(chronics_dir: str) -> Dict[str, Any]:
    meta_path = Path(chronics_dir) / "meta.json"
    if meta_path.exists():
        with meta_path.open() as fh:
            meta = json.load(fh)
        return meta
    ids = []
    for p in sorted(Path(chronics_dir).glob("seq_*.npy")):
        try:
            id_str = p.stem.split("_")[1]
            ids.append(int(id_str))
        except Exception:
            continue
    inferred = {'chronics': [{'id': i} for i in sorted(ids)]}
    return inferred

In [ ]:
META_INPUT = load_chronics_meta(CONFIG['data_dir'])
ALL_CHRONIC_IDS = [c['id'] for c in META_INPUT['chronics']]
if len(ALL_CHRONIC_IDS) == 0:
    raise SystemExit(f"No chronics found in {CONFIG['data_dir']}. Place seq_*.npy/tab_*.npy/lbl_*.npy there.")

In [ ]:
class GridRiskDataset(Dataset):
    def __init__(self, chronic_dir: str, chronic_ids: List[int], window_size: int, stride: int, horizon: int):
        self.window_size = int(window_size)
        self.stride = int(stride)
        self.horizon = int(horizon)
        self.windows = []
        self.data_cache = {}
        nan_count = 0

        print(f"Loading {len(chronic_ids)} chronics from {chronic_dir} with horizon={horizon}")
        for c_id in chronic_ids:
            seq_path = Path(chronic_dir) / f"seq_{c_id:04d}.npy"
            tab_path = Path(chronic_dir) / f"tab_{c_id:04d}.npy"
            lbl_path = Path(chronic_dir) / f"lbl_{c_id:04d}.npy"
            if not seq_path.exists() or not tab_path.exists() or not lbl_path.exists():
                print(f"  Skipping chronic {c_id}: missing one of seq/tab/lbl npy files.")
                continue
            seq = np.load(seq_path)
            tab = np.load(tab_path)
            lbl = np.load(lbl_path)

            nan_count += int(np.isnan(seq).sum() + np.isnan(tab).sum() + np.isnan(lbl).sum())
            
            seq_mask = ~np.isfinite(seq)
            if seq_mask.any():
                col_means = np.nanmean(seq, axis=0, keepdims=True)
                col_means = np.nan_to_num(col_means, nan=0.0)
                seq = np.where(seq_mask, np.repeat(col_means, seq.shape[0], axis=0), seq)

            tab = np.nan_to_num(tab, nan=np.nanmean(tab) if not np.isnan(tab).all() else 0.0)
            lbl = np.nan_to_num(lbl, nan=0.0)

            self.data_cache[c_id] = {'seq': seq.astype(np.float32),
                                     'tab': tab.astype(np.float32),
                                     'lbl': lbl.astype(np.float32)}

            n_steps = lbl.shape[0]
            max_start = n_steps - self.window_size - (self.horizon - 1)
            if max_start <= 0:
                continue
            for start in range(0, max_start, self.stride):
                self.windows.append((c_id, start))
        print(f"Loaded dataset: {len(self.windows):,} windows from {len(self.data_cache)} chronics. Scrubbed {nan_count:,} NaNs.")

    def __len__(self) -> int:
        return len(self.windows)

    def __getitem__(self, idx: int):
        c_id, start = self.windows[idx]
        end = start + self.window_size
        target_idx = end - 1 + self.horizon

        seq = self.data_cache[c_id]['seq'][start:end]        # shape [T, D]
        tab = self.data_cache[c_id]['tab'][end - 1]         # shape [features]
        target_seq = self.data_cache[c_id]['seq'][target_idx]

        # Label generation: thermal, voltage, blackout
        th_lbl = (target_seq[FEATURE_SLICES['rho'][0]:FEATURE_SLICES['rho'][1]] >= 0.85).astype(np.float32)
        v_or_slice = FEATURE_SLICES['v_or']
        v_ex_slice = FEATURE_SLICES['v_ex']
        v_lbl = ((target_seq[v_or_slice[0]:v_or_slice[1]] < 0.90) |
                 (target_seq[v_ex_slice[0]:v_ex_slice[1]] < 0.90)).astype(np.float32)
        bo_lbl = np.array([self.data_cache[c_id]['lbl'][target_idx][2]], dtype=np.float32)

        new_lbl = np.concatenate([th_lbl, v_lbl, bo_lbl]).astype(np.float32)

        # NORMALIZATION
        seq_clean = seq.copy().astype(np.float32)
        mean_scale = np.mean(seq_clean, axis=0, keepdims=True)
        std_scale = np.std(seq_clean, axis=0, keepdims=True) + 1e-8
        seq_clean = (seq_clean - mean_scale) / std_scale
        seq_clean = np.clip(seq_clean, -5.0, 5.0)

        # Safety
        if not np.isfinite(seq_clean).all():
            seq_clean = np.nan_to_num(seq_clean, nan=0.0, posinf=1e6, neginf=-1e6)

        # Return tensors (seq: [T, D], tab: [features], lbl: [targets])
        return torch.from_numpy(seq_clean.copy()), torch.from_numpy(tab.copy().astype(np.float32)), torch.from_numpy(new_lbl)

In [ ]:
example_cid = next(iter(ALL_CHRONIC_IDS))
example_seq = np.load(Path(CONFIG['data_dir']) / f"seq_{example_cid:04d}.npy")
SEQ_DIM = example_seq.shape[1]

meta_pre = META_INPUT
if 'feature_slices' in meta_pre:
    FEATURE_SLICES = {k: tuple(v) for k, v in meta_pre['feature_slices'].items()}
    SEQ_DIM = meta_pre.get('seq_dim', SEQ_DIM)
    N_LINE = meta_pre.get('n_line')
    N_LOAD = meta_pre.get('n_load')
    N_GEN = meta_pre.get('n_gen')
else:
    raise SystemExit("meta.json with 'feature_slices' and topology dims must be present in CHRONICS_DIR. Please produce using your collection script.")


META_OUT = {
    'feature_slices': {k: list(v) for k, v in FEATURE_SLICES.items()},
    'seq_dim': SEQ_DIM,
    'n_line': N_LINE,
    'n_load': N_LOAD,
    'n_gen': N_GEN,
    'pred_horizon': CONFIG['pred_horizon'],
    'chronics': META_INPUT['chronics'],
}
save_meta(META_OUT, os.path.join(CONFIG['save_dir'], "meta.json"))

In [7]:
def split_chronics(chronic_ids: List[int], train_frac: float, val_frac: float, seed: int) -> Tuple[List[int], List[int], List[int]]:
    ids = list(chronic_ids)
    rng = random.Random(seed)
    rng.shuffle(ids)
    n = len(ids)
    n_train = int(train_frac * n)
    n_val = int(val_frac * n)
    train = ids[:n_train]
    val = ids[n_train:n_train + n_val]
    test = ids[n_train + n_val:]
    return train, val, test

train_ids, val_ids, test_ids = split_chronics(ALL_CHRONIC_IDS, CONFIG['train_frac'], CONFIG['val_frac'], CONFIG['random_split_seed'])
print(f"Chronics split: train={len(train_ids)}, val={len(val_ids)}, test={len(test_ids)}")


WINDOW = CONFIG['window_size']
STRIDE = CONFIG['stride']
HORIZON = CONFIG['pred_horizon']

train_ds = GridRiskDataset(CONFIG['data_dir'], train_ids, WINDOW, STRIDE, HORIZON)
val_ds   = GridRiskDataset(CONFIG['data_dir'], val_ids,   WINDOW, STRIDE, HORIZON)
test_ds  = GridRiskDataset(CONFIG['data_dir'], test_ids,  WINDOW, STRIDE, HORIZON)

train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=16, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CONFIG['batch_size'] * 2, shuffle=False, num_workers=16, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=CONFIG['batch_size'] * 2, shuffle=False, num_workers=16, pin_memory=True)

print(f"Dataset sizes: Train {len(train_ds):,} | Val {len(val_ds):,} | Test {len(test_ds):,}")

Chronics split: train=220, val=47, test=48
Loading 220 chronics from /workspace/dataset/content/root/chronics with horizon=4
Loaded dataset: 51,459 windows from 220 chronics. Scrubbed 0 NaNs.
Loading 47 chronics from /workspace/dataset/content/root/chronics with horizon=4
Loaded dataset: 11,311 windows from 47 chronics. Scrubbed 0 NaNs.
Loading 48 chronics from /workspace/dataset/content/root/chronics with horizon=4
Loaded dataset: 9,988 windows from 48 chronics. Scrubbed 0 NaNs.
Dataset sizes: Train 51,459 | Val 11,311 | Test 9,988


In [ ]:
import torch
label_sums = torch.zeros(N_LINE * 2 + 1)
label_count = 0
for _, _, batch_lbl in train_loader:
    label_sums += batch_lbl.sum(dim=0)
    label_count += batch_lbl.shape[0]

label_freq = (label_sums / label_count).clamp(min=1e-8)
# Cap pos_weight to avoid huge gradients
pos_weight_cpu = ((1.0 - label_freq) / label_freq).clamp(min=1.0, max=10.0)
print(f"Pos weight summary: mean thermal={pos_weight_cpu[:N_LINE].mean():.2f}, mean voltage={pos_weight_cpu[N_LINE:2*N_LINE].mean():.2f}, blackout={pos_weight_cpu[-1]:.2f}")
print(f"Label Frequencies: mean thermal={label_freq[:N_LINE].mean():.4f}, mean voltage={label_freq[N_LINE:2*N_LINE].mean():.4f}, blackout={label_freq[-1]:.4f}")

pos_weight_cpu = pos_weight_cpu.to(torch.float32)

Pos weight summary: mean thermal=9.99, mean voltage=9.34, blackout=10.00
Label Frequencies: mean thermal=0.0044, mean voltage=0.0288, blackout=0.0001


In [8]:
from mamba_ssm import Mamba2

In [ ]:

class S6Block(nn.Module):
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
        super().__init__()
        self.n_line = n_line
        self.input_proj = nn.Sequential(nn.LayerNorm(seq_dim), nn.Linear(seq_dim, d_model), nn.LayerNorm(d_model))
        
        self.backbone = nn.ModuleList([Mamba2(d_model=d_model, d_state=d_state, d_conv=d_conv) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.class_head = nn.Sequential(nn.Linear(d_model, 128), nn.GELU(), nn.Dropout(0.2), nn.Linear(128, n_line * 2 + 1))
        # Expanded physics head: Rho (n_line), Gen (1), Load (1), Voltage_or (n_line)
        self.physics_head = nn.Sequential(nn.Linear(d_model, 128), nn.GELU(), nn.Linear(128, n_line + 2 + n_line))
        self.d_model = d_model

    def forward(self, x):
        residual = x
        x = self.norm(x)
        B, T, _ = x.shape
        xz = self.in_proj(x)
        x_branch, z = xz.chunk(2, dim=-1)
        x_branch = self.conv1d(x_branch.transpose(1, 2))[:, :, :T].transpose(1, 2)
        x_branch = self.act(x_branch)
        x_proj = self.x_proj(x_branch)
        B_s = x_proj[:, :, :self.d_state]
        C_s = x_proj[:, :, self.d_state:2*self.d_state]
        dt = F.softplus(self.dt_proj(x_proj[:, :, -1:]))
        A = -torch.exp(self.A_log.clamp(max=5.0))
        h = torch.zeros(B, self.d_inner, self.d_state, device=x.device)
        ys = []
        for t in range(T):
            exponent = (dt[:, t, :].unsqueeze(-1) * A).clamp(-20.0, 0.0)
            dA = torch.exp(exponent)
            dB = dt[:, t, :].unsqueeze(-1) * B_s[:, t, :].unsqueeze(1)
            h = h * dA + dB * x_branch[:, t, :].unsqueeze(-1)
            y = (h * C_s[:, t, :].unsqueeze(1)).sum(-1)
            ys.append(y)
        y = torch.stack(ys, dim=1)
        y = y * self.act(z)
        return self.out_proj(y) + residual

class GridRiskMamba(nn.Module):
    def __init__(self, seq_dim, n_line, n_load, n_gen, d_model=128, n_layers=4, d_state=32):
        super().__init__()
        self.n_line = n_line
        self.input_proj = nn.Sequential(nn.LayerNorm(seq_dim), nn.Linear(seq_dim, d_model), nn.LayerNorm(d_model))

        use_mamba = False
        try:
            from mamba_ssm import Mamba3
            if torch.cuda.is_available():
                use_mamba = True
        except Exception:
            use_mamba = False

        if use_mamba:
            self.backbone = nn.ModuleList([Mamba2(d_model=128, d_state=32, headdim=64) for _ in range(n_layers)])
            print("Using Mamba3 backbone (CUDA).") # Meant to be Mamba2
        else:
            self.backbone = nn.ModuleList([S6Block(d_model=d_model, d_state=d_state, d_conv=4, expand=2) for _ in range(n_layers)])
            print("Using S6Block PyTorch fallback backbone.")

        self.norm = nn.LayerNorm(d_model)
        # Risk classification head: Overload (n_line), Voltage Sag (n_line), Blackout(1)
        self.class_head = nn.Sequential(nn.Linear(d_model, 128), nn.GELU(), nn.Dropout(0.2), nn.Linear(128, n_line * 2 + 1))
        # Expanded physics head: Rho (n_line), Gen (1), Load (1), Voltage_or (n_line)
        self.physics_head = nn.Sequential(nn.Linear(d_model, 128), nn.GELU(), nn.Linear(128, n_line + 2 + n_line))
        self.d_model = d_model

    def forward(self, x):
        x = self.input_proj(x)
        for layer in self.backbone:
            x = layer(x)
        x = self.norm(x)
        embedding = x[:, -1, :]
        logits = self.class_head(embedding)
        physics_pred = self.physics_head(embedding)
        return logits, embedding, physics_pred

class PhysicsInformedLoss(nn.Module):
    def __init__(self, pos_weight, feature_slices, n_line, lam_kirchhoff=0.01, lam_thermal=0.01):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        self.lam_k = lam_kirchhoff
        self.lam_t = lam_thermal
        self.slices = feature_slices
        self.n_line = n_line

    def forward(self, logits, labels, physics_pred, raw_window):
        loss_cls = self.bce(logits, labels)
        
        pred_rho = physics_pred[:, :self.n_line]
        pred_gen_sum = physics_pred[:, self.n_line]
        pred_load_sum = physics_pred[:, self.n_line + 1]
        pred_v_or = physics_pred[:, self.n_line + 2:]
        
        # Power Balance Approximation (Kirchhoff proxy)
        loss_kirchhoff = F.smooth_l1_loss(pred_gen_sum, pred_load_sum)
        
        # Thermal & Voltage boundary penalties
        loss_thermal = torch.relu(-pred_rho).mean() + torch.relu(pred_rho - 2.0).mean()
        loss_voltage_bound = torch.relu(0.8 - pred_v_or).mean() + torch.relu(pred_v_or - 1.2).mean()

        s = self.slices
        actual_rho = raw_window[:, -1, s['rho'][0]:s['rho'][1]]
        actual_gen = raw_window[:, -1, s['gen_p'][0]:s['gen_p'][1]] / 1000.0
        actual_load = raw_window[:, -1, s['load_p'][0]:s['load_p'][1]] / 1000.0
        actual_v_or = raw_window[:, -1, s['v_or'][0]:s['v_or'][1]]

        loss_regression = (
            F.smooth_l1_loss(pred_rho, actual_rho)
            + F.smooth_l1_loss(pred_gen_sum, actual_gen.sum(dim=1))
            + F.smooth_l1_loss(pred_load_sum, actual_load.sum(dim=1))
            + F.smooth_l1_loss(pred_v_or, actual_v_or)
        )

        total = loss_cls + self.lam_k * loss_kirchhoff + self.lam_t * loss_thermal + 0.1 * loss_voltage_bound + 0.1 * loss_regression
        if not torch.isfinite(total):
            total = loss_cls.clamp(max=100.0)

        metrics = {'cls': loss_cls.item(), 'kirchhoff': loss_kirchhoff.item(), 'thermal': loss_thermal.item(), 'regression': loss_regression.item()}
        return total, metrics

In [ ]:
accelerator = None
try:
    mixed = 'bf16' if torch.cuda.is_available() else 'no'
    accelerator = Accelerator(mixed_precision=mixed, gradient_accumulation_steps=CONFIG['grad_accum'])
except Exception as e:
    print("Accelerator init failed; proceeding without accelerator. Error:", e)
    accelerator = None

device = accelerator.device if accelerator else torch.device("cuda" if torch.cuda.is_available() else "cpu")
pos_weight = pos_weight_cpu.to(device)

criterion = PhysicsInformedLoss(pos_weight=pos_weight, feature_slices=FEATURE_SLICES, n_line=N_LINE, lam_kirchhoff=CONFIG['lam_kirchhoff'], lam_thermal=CONFIG['lam_thermal'])

model = GridRiskMamba(seq_dim=SEQ_DIM, n_line=N_LINE, n_load=N_LOAD, n_gen=N_GEN, d_model=CONFIG['d_model'], n_layers=CONFIG['n_layers'], d_state=CONFIG['d_state'])
optimizer = AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
scheduler = CosineAnnealingLR(optimizer, T_max=CONFIG['epochs'])

if accelerator:
    model, optimizer, train_loader, val_loader, test_loader, scheduler = accelerator.prepare(model, optimizer, train_loader, val_loader, test_loader, scheduler)
    criterion = criterion.to(accelerator.device)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
device = accelerator.device if accelerator else torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ---------- TRAIN / EVAL ----------
def train_one_epoch(model, loader, criterion, optimizer, accelerator, scheduler, debug=True):
    """
    Run one training epoch.

    Parameters:
    - model: nn.Module
    - loader: DataLoader
    - criterion: loss module returning (loss, metrics_dict)
    - optimizer: optimizer
    - accelerator: accelerate.Accelerator instance or None
    - scheduler: LR scheduler
    - debug: if True, fail-fast on non-finite loss and print batch diagnostics

    Returns:
    - avg: dict with averaged components and total loss and lr
    """
    model.train()
    total_loss = 0.0
    components = {'cls': 0.0, 'kirchhoff': 0.0, 'thermal': 0.0, 'regression': 0.0}
    n_batches = 0
    nan_skips = 0

    # Determine device when not using accelerator
    if accelerator:
        device = accelerator.device
    else:
        try:
            device = next(model.parameters()).device
        except StopIteration:
            device = torch.device("cpu")

    for seq, tab, lbl in loader:
        if not accelerator:
            seq = seq.to(device=device, dtype=torch.float32)
            tab = tab.to(device=device, dtype=torch.float32)
            lbl = lbl.to(device=device, dtype=torch.float32)

        if accelerator:
            # Use accumulation if configured in accelerator
            with accelerator.accumulate(model):
                logits, embedding, physics_pred = model(seq)
                loss, metrics = criterion(logits, lbl, physics_pred, seq)

                if not torch.isfinite(loss):
                    nan_skips += 1
                    # Diagnostics + fail-fast (when debugging)
                    if debug:
                        print("NON-FINITE LOSS DETECTED (ACCELERATED). Batch diagnostics:")
                        try:
                            print("  seq finite:", torch.isfinite(seq).all().item(),
                                  "seq min/max:", seq.min().item(), seq.max().item(),
                                  "seq mean/std:", seq.mean().item(), seq.std().item())
                        except Exception:
                            print("  seq stats unavailable")
                        try:
                            print("  logits finite:", torch.isfinite(logits).all().item(),
                                  "logits mean/std:", logits.mean().item(), logits.std().item())
                            print("  logits (first 10):", logits[0, :min(10, logits.shape[1])].detach().cpu().numpy())
                        except Exception:
                            print("  logits stats unavailable")
                        raise RuntimeError("Non-finite loss; aborting to inspect batch.")
                    else:
                        optimizer.zero_grad()
                        continue

                accelerator.backward(loss)
                if accelerator.sync_gradients:
                    accelerator.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad()
        else:
            # Standard non-accelerated training path
            logits, embedding, physics_pred = model(seq)
            loss, metrics = criterion(logits, lbl, physics_pred, seq)

            if not torch.isfinite(loss):
                nan_skips += 1
                if debug:
                    print("NON-FINITE LOSS DETECTED. Batch diagnostics:")
                    try:
                        print("  seq finite:", torch.isfinite(seq).all().item(),
                              "seq min/max:", seq.min().item(), seq.max().item(),
                              "seq mean/std:", seq.mean().item(), seq.std().item())
                    except Exception:
                        print("  seq stats unavailable")
                    try:
                        print("  logits finite:", torch.isfinite(logits).all().item(),
                              "logits mean/std:", logits.mean().item(), logits.std().item())
                        print("  logits (first 10):", logits[0, :min(10, logits.shape[1])].detach().cpu().numpy())
                    except Exception:
                        print("  logits stats unavailable")
                    raise RuntimeError("Non-finite loss; aborting to inspect batch.")
                else:
                    optimizer.zero_grad()
                    continue

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()

        # Accumulate stats
        total_loss += float(loss.item())
        for k in components:
            components[k] += metrics.get(k, 0.0)
        n_batches += 1

        # Light per-batch debug logging for first few batches
        if debug and n_batches <= 5:
            try:
                print(f"batch {n_batches}: loss={loss.item():.6f}, logits_mean={logits.mean().item():.4f}, logits_std={logits.std().item():.4f}")
            except Exception:
                pass

    if nan_skips > 0:
        print(f"Skipped {nan_skips} non-finite batches.")

    # Step the scheduler
    try:
        scheduler.step()
    except Exception:
        pass

    avg = {k: components[k] / max(1, n_batches) for k in components}
    avg['total'] = total_loss / max(1, n_batches)
    avg['lr'] = scheduler.get_last_lr()[0] if hasattr(scheduler, 'get_last_lr') else optimizer.param_groups[0]['lr']
    return avg

@torch.no_grad()
def evaluate(model, loader, accelerator):
    model.eval()
    all_preds, all_true = [], []
    for seq, tab, lbl in loader:
        logits, _, _ = model(seq)
        probs = torch.sigmoid(logits)
        if accelerator:
            all_preds.append(accelerator.gather_for_metrics(probs).cpu().numpy())
            all_true.append(accelerator.gather_for_metrics(lbl).cpu().numpy())
        else:
            all_preds.append(probs.cpu().numpy())
            all_true.append(lbl.cpu().numpy())

    preds = np.concatenate(all_preds, axis=0)
    true = np.concatenate(all_true, axis=0)
    preds = np.nan_to_num(preds, nan=0.5)
    true = np.nan_to_num(true, nan=0.0)

    def get_metrics(t, p):
        if t.sum() == 0: 
            return {'auroc': 0.0, 'ap': 0.0, 'f1': 0.0, 'acc': 0.0}
        pb = (p >= 0.5).astype(int)
        return {'auroc': roc_auc_score(t, p), 'ap': average_precision_score(t, p), 'f1': f1_score(t, pb, zero_division=0), 'acc': accuracy_score(t, pb)}

    results = {}
    results['Thermal'] = get_metrics(true[:, :N_LINE].ravel(), preds[:, :N_LINE].ravel())
    results['Voltage'] = get_metrics(true[:, N_LINE:2*N_LINE].ravel(), preds[:, N_LINE:2*N_LINE].ravel())
    results['Blackout'] = get_metrics(true[:, -1], preds[:, -1])

    results['mean_auroc'] = np.mean([v['auroc'] for k, v in results.items() if isinstance(v, dict)])
    results['mean_ap'] = np.mean([v['ap'] for k, v in results.items() if isinstance(v, dict)])
    results['mean_acc'] = np.mean([v['acc'] for k, v in results.items() if isinstance(v, dict)])
    return results, preds, true

Using Mamba3 backbone (CUDA).
Model parameters: 584,369
Device: cuda


In [ ]:
def run_training():
    print("="*40)
    print("TRAINING START")
    print("="*40)
    best_val_ap = -1.0
    patience = CONFIG['patience']
    patience_counter = 0
    history = []
    for epoch in range(CONFIG['epochs']):
        t0 = time.time()
        train_metrics = train_one_epoch(model, train_loader, criterion, optimizer, accelerator, scheduler)
        val_results, _, _ = evaluate(model, val_loader, accelerator)
        elapsed = time.time() - t0
        mean_ap = val_results['mean_ap']
        print(f"Epoch {epoch+1}/{CONFIG['epochs']} | Train loss {train_metrics['total']:.4f} | Val mean AP {mean_ap:.4f}| ACC: {val_results['mean_acc']:.4f} | AUROC: {val_results['mean_auroc']:.4f} | time {elapsed:.0f}s")
        if mean_ap > best_val_ap: # Compare best model by mean AP
            best_val_ap = mean_ap
            patience_counter = 0
            unwrapped = accelerator.unwrap_model(model) if accelerator else model
            torch.save({'model_state': unwrapped.state_dict(), 'epoch': epoch+1, 'best_ap': best_val_ap, 'config': CONFIG}, os.path.join(CONFIG['save_dir'], 'best_model.pt'))
            print(f"Saved best model (AP={best_val_ap:.4f})")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break
        history.append({'epoch': epoch+1, 'train_loss': train_metrics['total'], 'val_mean_ap': mean_ap})
    with open(os.path.join(CONFIG['save_dir'], 'history.json'), 'w') as fh:
        json.dump(history, fh, indent=2)
    print("TRAINING COMPLETE. Best val AP:", best_val_ap)

# ---------- EMBEDDING EXTRACTION ----------
@torch.no_grad()
def extract_all_embeddings(model, loader, accelerator):
    model.eval()
    all_emb, all_tab, all_lbl = [], [], []
    for seq, tab, lbl in loader:
        _, embedding, _ = model(seq)
        if accelerator:
            all_emb.append(accelerator.gather_for_metrics(embedding).cpu().numpy())
            all_tab.append(accelerator.gather_for_metrics(tab).cpu().numpy())
            all_lbl.append(accelerator.gather_for_metrics(lbl).cpu().numpy())
        else:
            all_emb.append(embedding.cpu().numpy())
            all_tab.append(tab.cpu().numpy())
            all_lbl.append(lbl.cpu().numpy())
    return np.concatenate(all_emb, axis=0), np.concatenate(all_tab, axis=0), np.concatenate(all_lbl, axis=0)

def save_embeddings_and_labels():
    print("Extracting embeddings (train/val/test)...")
    train_emb, train_tab, train_lbl = extract_all_embeddings(model, train_loader, accelerator)
    val_emb, val_tab, val_lbl = extract_all_embeddings(model, val_loader, accelerator)
    test_emb, test_tab, test_lbl = extract_all_embeddings(model, test_loader, accelerator)

    X_train = np.concatenate([train_emb, train_tab], axis=1)
    X_val = np.concatenate([val_emb, val_tab], axis=1)
    X_test = np.concatenate([test_emb, test_tab], axis=1)

    np.save(os.path.join(CONFIG['save_dir'], 'X_train.npy'), X_train)
    np.save(os.path.join(CONFIG['save_dir'], 'X_val.npy'), X_val)
    np.save(os.path.join(CONFIG['save_dir'], 'X_test.npy'), X_test)
    np.save(os.path.join(CONFIG['save_dir'], 'y_train.npy'), train_lbl)
    np.save(os.path.join(CONFIG['save_dir'], 'y_val.npy'), val_lbl)
    np.save(os.path.join(CONFIG['save_dir'], 'y_test.npy'), test_lbl)
    print("Saved embeddings and labels to", CONFIG['save_dir'])

In [ ]:
def run_xgboost_phase2():
    print("\n" + "="*40)
    print("PHASE 2: XGBOOST ENSEMBLE TRAINING")
    print("="*40)
    import pickle
    from xgboost import XGBClassifier
    from sklearn.multioutput import MultiOutputClassifier
    from sklearn.calibration import CalibratedClassifierCV
    from sklearn.metrics import average_precision_score

    MODEL_DIR = CONFIG['save_dir']
    
    print("Loading extracted embeddings from disk...")
    X_train_xgb = np.load(os.path.join(MODEL_DIR, "X_train.npy"))
    y_train_xgb = np.load(os.path.join(MODEL_DIR, "y_train.npy"))
    X_test_xgb  = np.load(os.path.join(MODEL_DIR, "X_test.npy"))
    y_test_xgb  = np.load(os.path.join(MODEL_DIR, "y_test.npy"))
    
    print(f"XGBoost Dimensions: X_train {X_train_xgb.shape}, y_train {y_train_xgb.shape}")
    
    params = {
        'n_estimators': 200,
        'max_depth': 6,
        'objective': 'binary:logistic',
        'verbosity': 1,
        'eval_metric': 'logloss',
    }
    
    if torch.cuda.is_available():
        print("CUDA detected: forcing XGBoost to GPU backend (device='cuda').")
        params.update({'tree_method': 'hist', 'device': 'cuda'})
    else:
        print("CUDA not detected: using CPU backend.")
        
    print("Initializing Correlation-Aware Classifier Chain XGBoost Stack...")
    raw_xgb = XGBClassifier(**params)
    
    # Use ClassifierChain
    from sklearn.multioutput import ClassifierChain
    model_xgb = ClassifierChain(raw_xgb, order='random', random_state=42)
    
    print("Training XGBoost (Fitting one calibrated classifier per physical target)...")
    model_xgb.fit(X_train_xgb, y_train_xgb)
    
    print("Evaluating Macro Average Precision on Test Set...")
    preds_xgb = model_xgb.predict_proba(X_test_xgb)
    macro_ap = average_precision_score(y_test_xgb, preds_xgb, average='macro')
    print(f"✅ XGBoost Evaluation Complete. Macro AP: {macro_ap:.4f}")
    
    out_path = os.path.join(MODEL_DIR, "xgb_model.pkl")
    with open(out_path, "wb") as fh:
        pickle.dump(model_xgb, fh)
    print(f"💾 Exported calibrated XGBoost gateway to {out_path}")

In [12]:
def smoke_checks():
    # Basic shape checks
    seq_batch, tab_batch, lbl_batch = next(iter(train_loader))
    assert seq_batch.ndim == 3, "Seq batch must be [B, T, D]"
    assert tab_batch.ndim == 2, "Tab batch must be [B, features]"
    assert lbl_batch.ndim == 2, "Labels must be [B, targets]"
    print("SMOKE CHECKS PASS: shapes look good.")

In [13]:
smoke_checks()
print("\nAll systems nominal. Initiating A100 Training Pipeline...")

SMOKE CHECKS PASS: shapes look good.

All systems nominal. Initiating A100 Training Pipeline...


In [14]:
run_training()
print("\nSaving optimal embeddings for XGBoost Phase 2...")
save_embeddings_and_labels()
print("\n✅ PHASE 1 PIPELINE COMPLETION PROTOCOL VERIFIED.")

TRAINING START
batch 1: loss=0.943666, logits_mean=-0.0134, logits_std=0.2259
batch 2: loss=0.923674, logits_mean=-0.0497, logits_std=0.2333
batch 3: loss=0.902204, logits_mean=-0.0827, logits_std=0.2528
batch 4: loss=0.877919, logits_mean=-0.1141, logits_std=0.2715
batch 5: loss=0.870854, logits_mean=-0.1461, logits_std=0.2925
Epoch 1/20 | Train loss 0.2779 | Val mean AP 0.4004| ACC: 0.9909 | AUROC: 0.9691 | time 100s
Saved best model (AP=0.4004)
batch 1: loss=0.146090, logits_mean=-5.3106, logits_std=1.9580
batch 2: loss=0.132662, logits_mean=-5.2981, logits_std=2.0008
batch 3: loss=0.147099, logits_mean=-5.3689, logits_std=2.0457
batch 4: loss=0.138220, logits_mean=-5.4598, logits_std=2.0267
batch 5: loss=0.143740, logits_mean=-5.4743, logits_std=2.0428
Epoch 2/20 | Train loss 0.1153 | Val mean AP 0.4974| ACC: 0.9931 | AUROC: 0.9696 | time 18s
Saved best model (AP=0.4974)
batch 1: loss=0.103686, logits_mean=-6.8243, logits_std=2.4686
batch 2: loss=0.105203, logits_mean=-7.0826, logi

In [ ]:
# Trigger Phase 2 Immediately
run_xgboost_phase2()
        
print("\n🏆 FULL ML TRAINING PIPELINE (MAMBA + XGBOOST) COMPLETED SUCCESSFULLY.")
        
# Safely copy into a dedicated folder on Drive to prevent FileExistsError crashes
shutil.make_archive('FINAL_GRID_MODELS_BACKUP', 'zip', CONFIG['save_dir'])
print("\n💾 All assets securely backed up to Google Drive.")


PHASE 2: XGBOOST ENSEMBLE TRAINING
Loading extracted embeddings from disk...
XGBoost Dimensions: X_train (51459, 154), y_train (51459, 119)
CUDA detected: forcing XGBoost to GPU backend (device='cuda').
Initializing Correlation-Aware Classifier Chain XGBoost Stack...
Training XGBoost (Fitting one calibrated classifier per physical target)...
Evaluating Macro Average Precision on Test Set...


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1192: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1192: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1192: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1192: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1192: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1192: UserWarning: No positive c

✅ XGBoost Evaluation Complete. Macro AP: 0.2316
💾 Exported calibrated XGBoost gateway to /workspace/grid_models/xgb_model.pkl

🏆 FULL ML TRAINING PIPELINE (MAMBA + XGBOOST) COMPLETED SUCCESSFULLY.

💾 All assets securely backed up to Google Drive.


## Stateless XGB Classifier Baseline

In [ ]:
import os
import numpy as np
from xgboost import XGBClassifier
from sklearn.multioutput import ClassifierChain
from sklearn.metrics import average_precision_score

print("Extracting Stateless raw data (Timestep T) from Python memory...")

def extract_stateless_X(dataset):
    raw_X = []
    for idx in range(len(dataset)):
        c_id, start = dataset.windows[idx]
        end = start + dataset.window_size
        
        # Grab strictly the grid's physics (650 dims) and tab at the 'current' time T
        raw_physics = dataset.data_cache[c_id]['seq'][end - 1]
        tab = dataset.data_cache[c_id]['tab'][end - 1]
        
        raw_X.append(np.concatenate([raw_physics, tab]))
    return np.array(raw_X, dtype=np.float32)

X_train_stateless = extract_stateless_X(train_ds)
X_test_stateless = extract_stateless_X(test_ds)
y_train_stateless = np.load(os.path.join(CONFIG['save_dir'], 'y_train.npy'))
y_test_stateless  = np.load(os.path.join(CONFIG['save_dir'], 'y_test.npy'))

print(f"Stateless Arrays Built! X_train shape: {X_train_stateless.shape}")

print("\n" + "="*40)
print("TRAINING STATELESS BASELINE (No Time-Series Momentum)")
print("="*40)

params = {'n_estimators': 200, 'max_depth': 6, 'objective': 'binary:logistic', 'verbosity': 1}
import torch
if torch.cuda.is_available():
    params.update({'tree_method': 'hist', 'device': 'cuda'})

stateless_base = XGBClassifier(**params)
model_stateless = ClassifierChain(stateless_base, order='random', random_state=42)

print("Fitting correlation-aware sequence... this might take a minute...")
model_stateless.fit(X_train_stateless, y_train_stateless)

print("Evaluating Stateless Average Precision...")
preds_stateless = model_stateless.predict_proba(X_test_stateless)

N_L = (y_test_stateless.shape[1] - 1) // 2
th_ap = average_precision_score(y_test_stateless[:, :N_L].ravel(), preds_stateless[:, :N_L].ravel())
v_ap = average_precision_score(y_test_stateless[:, N_L:2*N_L].ravel(), preds_stateless[:, N_L:2*N_L].ravel())
bo_ap = average_precision_score(y_test_stateless[:, -1], preds_stateless[:, -1]) if y_test_stateless[:, -1].sum() > 0 else 0.0

stateless_mean_ap = (th_ap + v_ap + bo_ap) / 3.0

print(f"\n❌ STATELESS BASELINE TRUE MEAN AP: {stateless_mean_ap:.4f}")
print("Compare this output to your Mamba's AP!")

Extracting Stateless raw data (Timestep T) from Python memory...
Stateless Arrays Built! X_train shape: (51459, 675)

TRAINING STATELESS BASELINE (No Time-Series Momentum)
Fitting correlation-aware sequence... this might take a minute...
Evaluating Stateless Average Precision...

❌ STATELESS BASELINE TRUE MEAN AP: 0.0059
Compare this output to your Mamba's AP!


In [21]:
stateless_macro_ap = average_precision_score(y_test_stateless, preds_stateless, average='macro')
print(f"\n❌ STATELESS BASELINE MACRO AP: {stateless_macro_ap:.4f}")

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1192: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1192: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1192: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1192: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1192: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1192: UserWarning: No positive c


❌ STATELESS BASELINE MACRO AP: 0.0128


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1192: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1192: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1192: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1192: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1192: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1192: UserWarning: No positive c

In [22]:
print(f"\n❌ STATELESS BASELINE MACRO AP: {stateless_macro_ap:.4f}")


❌ STATELESS BASELINE MACRO AP: 0.0128


## Transformer-Mamba-XGB Model Modification

In [23]:
class GridRiskMambaTF(nn.Module):
    def __init__(self, seq_dim, n_line, n_load, n_gen, d_model=128, n_layers=4, d_state=32):
        super().__init__()
        self.n_line = n_line
        self.input_proj = nn.Sequential(nn.LayerNorm(seq_dim), nn.Linear(seq_dim, d_model), nn.LayerNorm(d_model))

        # Prefer Mamba3 on CUDA if available
        use_mamba = False
        try:
            from mamba_ssm import Mamba2
            if torch.cuda.is_available():
                use_mamba = True
        except Exception:
            use_mamba = False

        if use_mamba:
            self.backbone = nn.ModuleList([Mamba2(d_model=128, d_state=32, headdim=64) for _ in range(n_layers)])
            print("Using Mamba3 backbone (CUDA).")
        else:
            self.backbone = nn.ModuleList([S6Block(d_model=d_model, d_state=d_state, d_conv=4, expand=2) for _ in range(n_layers)])
            print("Using S6Block PyTorch fallback backbone.")

        self.norm = nn.LayerNorm(d_model)
        self.class_head = nn.Sequential(nn.Linear(d_model, 128), nn.GELU(), nn.Dropout(0.2), nn.Linear(128, n_line * 2 + 1))
        # Expanded physics head: Rho (n_line), Gen (1), Load (1), Voltage_or (n_line)
        self.physics_head = nn.Sequential(nn.Linear(d_model, 128), nn.GELU(), nn.Linear(128, n_line + 2 + n_line))
        self.d_model = d_model
        # Transformer Layers
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=4, dim_feedforward=256, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)

    def forward(self, x):
        x = self.input_proj(x)
        x = self.transformer(x)
        for layer in self.backbone:
            x = layer(x)
        x = self.norm(x)
        embedding = x[:, -1, :]
        logits = self.class_head(embedding)
        physics_pred = self.physics_head(embedding)
        return logits, embedding, physics_pred

In [24]:
import os
CONFIG['save_dir'] = './grid_models_TRANSFORMER'
os.makedirs(CONFIG['save_dir'], exist_ok=True)

accelerator = None
try:
    # Mixed precision: bf16 on GPUs that support it; else fp32
    mixed = 'bf16' if torch.cuda.is_available() else 'no'
    accelerator = Accelerator(mixed_precision=mixed, gradient_accumulation_steps=CONFIG['grad_accum'])
except Exception as e:
    print("Accelerator init failed; proceeding without accelerator. Error:", e)
    accelerator = None

device = accelerator.device if accelerator else torch.device("cuda" if torch.cuda.is_available() else "cpu")
pos_weight = pos_weight_cpu.to(device)

criterion = PhysicsInformedLoss(pos_weight=pos_weight, feature_slices=FEATURE_SLICES, n_line=N_LINE, lam_kirchhoff=CONFIG['lam_kirchhoff'], lam_thermal=CONFIG['lam_thermal'])

model = GridRiskMambaTF(seq_dim=SEQ_DIM, n_line=N_LINE, n_load=N_LOAD, n_gen=N_GEN, d_model=CONFIG['d_model'], n_layers=CONFIG['n_layers'], d_state=CONFIG['d_state'])
optimizer = AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
scheduler = CosineAnnealingLR(optimizer, T_max=CONFIG['epochs'])

if accelerator:
    model, optimizer, train_loader, val_loader, test_loader, scheduler = accelerator.prepare(model, optimizer, train_loader, val_loader, test_loader, scheduler)
    criterion = criterion.to(accelerator.device)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
device = accelerator.device if accelerator else torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Using Mamba3 backbone (CUDA).
Model parameters: 849,329
Device: cuda


In [25]:
run_training()
save_embeddings_and_labels()

TRAINING START
batch 1: loss=0.957343, logits_mean=0.0013, logits_std=0.2304
batch 2: loss=0.930588, logits_mean=-0.0347, logits_std=0.2220
batch 3: loss=0.911581, logits_mean=-0.0617, logits_std=0.2276
batch 4: loss=0.892972, logits_mean=-0.0932, logits_std=0.2500
batch 5: loss=0.882533, logits_mean=-0.1209, logits_std=0.2698
Epoch 1/20 | Train loss 0.2884 | Val mean AP 0.3354| ACC: 0.9905 | AUROC: 0.9808 | time 103s
Saved best model (AP=0.3354)
batch 1: loss=0.156003, logits_mean=-5.3410, logits_std=1.9045
batch 2: loss=0.155309, logits_mean=-5.4377, logits_std=2.0683
batch 3: loss=0.148769, logits_mean=-5.5907, logits_std=2.0785
batch 4: loss=0.149558, logits_mean=-5.5277, logits_std=2.0567
batch 5: loss=0.144250, logits_mean=-5.4904, logits_std=2.0162
Epoch 2/20 | Train loss 0.1231 | Val mean AP 0.5107| ACC: 0.9933 | AUROC: 0.9576 | time 26s
Saved best model (AP=0.5107)
batch 1: loss=0.093295, logits_mean=-6.9465, logits_std=2.3640
batch 2: loss=0.102150, logits_mean=-7.0261, logit

In [26]:
run_xgboost_phase2()
shutil.make_archive('FINAL_GRID_MODELS_BACKUP', 'zip', CONFIG['save_dir'])


PHASE 2: XGBOOST ENSEMBLE TRAINING
Loading extracted embeddings from disk...
XGBoost Dimensions: X_train (51459, 154), y_train (51459, 119)
CUDA detected: forcing XGBoost to GPU backend (device='cuda').
Initializing Correlation-Aware Classifier Chain XGBoost Stack...
Training XGBoost (Fitting one calibrated classifier per physical target)...
Evaluating Macro Average Precision on Test Set...


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1192: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1192: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1192: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1192: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1192: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1192: UserWarning: No positive c

✅ XGBoost Evaluation Complete. Macro AP: 0.2399
💾 Exported calibrated XGBoost gateway to ./grid_models_TRANSFORMER/xgb_model.pkl


'/workspace/FINAL_GRID_MODELS_BACKUP.zip'